<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/phi_2_7_performance_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from google.colab import auth
import os
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Authenticate Google User
auth.authenticate_user()

# 3. Install necessary libraries
!pip install -q accelerate transformers datasets scikit-learn matplotlib bitsandbytes

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 40.8 MB/s eta 0:00:00
Using device: cuda
GPU: NVIDIA L4


In [ ]:
from torch.utils.data import Dataset
import pandas as pd

class TestDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def read_csv_file(file_path):
    try:
        data = pd.read_csv(file_path, names=['body', 'inflation'], header=0, dtype={'body': str, 'inflation': str})
        data = data.dropna(subset=['inflation'])
        return data
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# IMF Economist Prompt (Must match training exactly)
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post: {post}
Classification:"""

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=str(post))

In [ ]:
# Path to test data
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv'

# Read data
test_data = read_csv_file(file_path)

if test_data is not None:
    # Apply prompt formatting
    test_data['formatted_body'] = test_data['body'].apply(format_with_prompt)
    print(f"Data loaded successfully. Number of samples: {len(test_data)}")
else:
    raise ValueError("Failed to load test data.")

Data loaded successfully. Number of samples: 200


In [ ]:
import os
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

checkpoint_path = '/content/drive/MyDrive/world-inflation/data/model/Phi-2-fine-tuning-rerun/checkpoint-343/'
base_model_name = "microsoft/phi-2"

print(f"Target Checkpoint: {checkpoint_path}")

# 1. Load Tokenizer (From Base Model)
print(f"Loading tokenizer from {base_model_name}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    raise

# 2. Load Configuration (From Base Model)
print(f"Loading configuration from {base_model_name}...")
try:
    config = AutoConfig.from_pretrained(
        base_model_name,
        num_labels=3,
        trust_remote_code=True
    )
except Exception as e:
    print(f"Error loading configuration: {e}")
    raise

# 3. Load Model Weights (From Local Checkpoint-343)
print(f"Loading model weights from {checkpoint_path}...")
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path,
        config=config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        local_files_only=True
    )

    model.eval()
    print("Successfully loaded Phi-2 model (Epoch 7) from checkpoint!")

except OSError as e:
    print(f"\n[Error] Failed to load model weights: {e}")
    if os.path.exists(checkpoint_path):
        print(f"Files found in {checkpoint_path}:")
        print(os.listdir(checkpoint_path))
    else:
        print(f"Directory not found: {checkpoint_path}")
    raise

Target Checkpoint: /content/drive/MyDrive/world-inflation/data/model/Phi-2-fine-tuning-rerun/checkpoint-343/
Loading tokenizer from microsoft/phi-2...
Loading configuration from microsoft/phi-2...


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

Loading model weights from /content/drive/MyDrive/world-inflation/data/model/Phi-2-fine-tuning-rerun/checkpoint-343/...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Successfully loaded Phi-2 model (Epoch 7) from checkpoint!


In [ ]:
from torch.utils.data import DataLoader

if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id
    print(f"Fixed: Model pad_token_id set to {model.config.pad_token_id}")

# Resize token embeddings if necessary (in case tokenizer size changed)
model.resize_token_embeddings(len(tokenizer))

# 1. Tokenize
print("Tokenizing test data...")
test_encodings = tokenizer(
    test_data['formatted_body'].tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

# 2. Prepare Labels
try:
    test_labels = test_data['inflation'].astype(int).tolist()
except ValueError:
    print("Warning: Found non-numeric labels. coercing...")
    test_data['inflation'] = pd.to_numeric(test_data['inflation'], errors='coerce')
    test_data = test_data.dropna(subset=['inflation'])
    test_labels = test_data['inflation'].astype(int).tolist()

# 3. Create DataLoader
test_dataset = TestDataset(test_encodings, test_labels)
test_loader = DataLoader(test_dataset, batch_size=8)

# 4. Inference Loop
print("Starting inference...")
true_labels = []
predicted_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        inputs = {key: val.to(model.device) for key, val in batch.items() if key != 'labels'}
        labels = batch['labels'].to(model.device)

        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {(batch_idx + 1) * 8} / {len(test_dataset)} samples...")

print("Inference completed.")

Fixed: Model pad_token_id set to 50256
Tokenizing test data...
Starting inference...
Processed 40 / 200 samples...
Processed 80 / 200 samples...
Processed 120 / 200 samples...
Processed 160 / 200 samples...
Processed 200 / 200 samples...
Inference completed.


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix
import numpy as np

# Calculate Metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, average=None)
precision = precision_score(true_labels, predicted_labels, average=None)
f1 = f1_score(true_labels, predicted_labels, average=None)

# Display Classification Report
print("\n=== Classification Report ===")
print(classification_report(true_labels, predicted_labels, digits=4))

# Display Confusion Matrix
print("\n=== Confusion Matrix ===")
print(confusion_matrix(true_labels, predicted_labels))

# Display Detailed Metrics Table
print("\n=== Detailed Metrics ===")
print("+--------------+-----------+----------+----------+----------+")
print("|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |")
print("+--------------+-----------+----------+----------+----------+")
for i in range(len(recall)):
    print(f"| Class {i}      |    ---    |   {recall[i]:.4f} |   {precision[i]:.4f} |   {f1[i]:.4f} |")
print("+--------------+-----------+----------+----------+----------+")
print(f"| Macro Average|    ---    |   {np.mean(recall):.4f} |   {np.mean(precision):.4f} |   {np.mean(f1):.4f} |")
print("+--------------+-----------+----------+----------+----------+")
print(f"| Micro Average|   {accuracy:.4f}  |   {accuracy:.4f} |   {accuracy:.4f} |   {accuracy:.4f} |")
print("+--------------+-----------+----------+----------+----------+")

print(f"\nTotal test samples: {len(true_labels)}")


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7042    0.8065    0.7519        62
           1     0.7183    0.6623    0.6892        77
           2     0.7414    0.7049    0.7227        61

    accuracy                         0.7200       200
   macro avg     0.7213    0.7246    0.7213       200
weighted avg     0.7210    0.7200    0.7188       200


=== Confusion Matrix ===
[[50 11  1]
 [12 51 14]
 [ 9  9 43]]

=== Detailed Metrics ===
+--------------+-----------+----------+----------+----------+
|   Metric     | Accuracy  |  Recall  | Precision|  F1 Score |
+--------------+-----------+----------+----------+----------+
| Class 0      |    ---    |   0.8065 |   0.7042 |   0.7519 |
| Class 1      |    ---    |   0.6623 |   0.7183 |   0.6892 |
| Class 2      |    ---    |   0.7049 |   0.7414 |   0.7227 |
+--------------+-----------+----------+----------+----------+
| Macro Average|    ---    |   0.7246 |   0.7213 |   0.7213 |
+-